In [1]:
from __future__ import annotations
import os, json
from pathlib import Path
import pandas as pd


In [2]:
from __future__ import annotations
import json, re
from pathlib import Path

# === EDIT ME ===
DATASET_ROOT = Path("/Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/ambulance_dataset_fast_150_espisode_cpu_30_senario")
OUT_MANIFEST = DATASET_ROOT / "manifests" / "episodes.jsonl"
# ===============

def _resolve(base: Path, rel: str) -> Path:
    p = Path(rel)
    if p.is_absolute():
        return p
    cand = (base / p)
    if cand.exists():
        return cand.resolve()
    cand2 = (base.parent / p)
    return cand2.resolve() if cand2.exists() else cand.resolve()

def stem_to_episode_id(stem: str) -> str:
    # "20250928_014719-e8c7fd13_transitions" -> "20250928_014719-e8c7fd13"
    return re.sub(r"_transitions$", "", stem)

def main():
    OUT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    cons_idx = json.loads((DATASET_ROOT / "consolidated_index.json").read_text())
    lines = []

    for batch in cons_idx["batches"]:
        batch_dir = DATASET_ROOT / batch["storage_path"]
        batch_idx = json.loads((batch_dir / "index.json").read_text())
        base_path = batch_idx.get("base_path") or batch["storage_path"]
        base_dir  = DATASET_ROOT / base_path

        for scenario, entries in batch_idx["scenarios"].items():
            for e in entries:
                t_rel = e["transitions_file"]
                m_rel = e.get("metadata_file")
                t_abs = _resolve(base_dir, t_rel)
                m_abs = _resolve(base_dir, m_rel) if m_rel else None

                stem  = Path(t_rel).name.replace(".parquet", "")
                ep_id = stem_to_episode_id(stem)

                lines.append(json.dumps({
                    "scenario": scenario,
                    "episode_id": ep_id,
                    "transitions_path": str(t_abs),
                    "metadata_path": str(m_abs) if m_abs else None,
                    "batch": batch["storage_path"],
                }))

    OUT_MANIFEST.write_text("\n".join(lines) + "\n")
    print(f"[ok] wrote {len(lines)} episodes -> {OUT_MANIFEST}")

if __name__ == "__main__":
    main()


[ok] wrote 30 episodes -> /Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/ambulance_dataset_fast_150_espisode_cpu_30_senario/manifests/episodes.jsonl


In [ ]:
from __future__ import annotations
import os, json, math
from pathlib import Path
from typing import Dict, Any, Iterable
import pandas as pd
import numpy as np

# === EDIT ME ===
MANIFEST = Path("/Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/ambulance_dataset_fast_150_espisode_cpu_30_senario/manifests/episodes.jsonl")
OUT_DIR  = Path("/Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/ambulance_dataset_fast_150_espisode_cpu_30_senario/captions")
# ===============

OUT_DIR.mkdir(parents=True, exist_ok=True)

# thresholds for short event phrasing
TTC_DANGER  = float(os.getenv("CAP_TTC_DANGER", "0.7"))   # seconds
TTC_CAUTION = float(os.getenv("CAP_TTC_CAUTION", "3.0"))

# columns we may use; code tolerates missing ones
NUMERIC_COLS = [
    "speed", "ego_vx", "ego_vy", "lane_position", "min_ttc",
    "traffic_density", "vehicle_count", "average_speed", "speed_variance"
]
BOOL_COLS = ["ambulance_scenario", "done"]
STR_COLS  = ["scenario", "summary_text"]
AUX_COLS  = ["agent_id", "ambulance_agent_index"]

def _safe_float(x, default=0.0) -> float:
    try:
        v = float(x)
        if math.isfinite(v):
            return v
    except Exception:
        pass
    return float(default)

def _safe_int(x, default=0) -> int:
    try:
        v = int(x)
        return v
    except Exception:
        try:
            v = int(float(x))
            return v
        except Exception:
            return int(default)

def _scenario_nice(s: str) -> str:
    return str(s).replace("_", " ")

def _get(row: Dict[str, Any], keys: Iterable[str], default=None):
    for k in keys:
        if k in row and pd.notna(row[k]):
            return row[k]
    return default

def build_caption(row: Dict[str, Any]) -> str:
    """Deterministic, compact caption for the ambulance ego vehicle."""
    scen = _scenario_nice(_get(row, ["scenario"], "highway"))
    amb  = bool(_get(row, ["ambulance_scenario"], True))
    lane = _safe_int(_get(row, ["lane_position"], 0), 0)

    # prefer 'speed'; fall back to |vx| (approx forward)
    spd  = _safe_float(_get(row, ["speed", "ego_vx"], 0.0), 0.0)
    ttc  = _safe_float(_get(row, ["min_ttc"], 30.0), 30.0)
    dens = _safe_float(_get(row, ["traffic_density"], 0.0), 0.0)
    vehs = _safe_int(_get(row, ["vehicle_count"], 0), 0)

    role = "Ambulance (ego) with right-of-way" if amb else "Ego vehicle"
    base = (f"{role} in {scen}. lane={lane}, speed={spd:.1f}, ttc={ttc:.1f}s, "
            f"density={dens:.2f}, vehicles_nearby={vehs}. ")

    # tiny event phrase based on TTC
    if ttc <= TTC_DANGER:
        event = "Immediate hazard ahead; maintain corridor."
    elif ttc <= TTC_CAUTION:
        event = "Caution; prepare to yield corridor."
    else:
        event = "Other drivers should yield and open a corridor for the ambulance."
    return base + event

def process_episode(item: Dict[str, Any]) -> int:
    ep_id = item["episode_id"]
    t_path = Path(item["transitions_path"])
    if not t_path.exists():
        print(f"[warn] missing {t_path}")
        return 0

    # read minimally; keep step + selection columns
    cols = ["step"] + NUMERIC_COLS + BOOL_COLS + STR_COLS + AUX_COLS
    df = pd.read_parquet(t_path, columns=[c for c in cols if c in pd.read_parquet(t_path).columns])

    # ensure 'step'
    if "step" not in df.columns:
        df = df.reset_index().rename(columns={"index": "step"})

    # forward-fill ambulance_agent_index per episode if sparse
    if "ambulance_agent_index" in df.columns:
        # sometimes it's per-episode; forward/back fill within ep
        df["ambulance_agent_index"] = df["ambulance_agent_index"].ffill().bfill()

    # --- select ambulance agent rows (unique per step) ---
    if "agent_id" in df.columns and "ambulance_agent_index" in df.columns:
        # filter to ambulance rows
        amb_idx = df["ambulance_agent_index"]
        df = df[df["agent_id"] == amb_idx]
        # fallback: if nothing selected (rare), keep first agent per step
        if df.empty:
            df = pd.read_parquet(t_path, columns=[c for c in ["step","scenario","speed","ego_vx","min_ttc","traffic_density","vehicle_count","lane_position","ambulance_scenario"] if c in pd.read_parquet(t_path).columns])
            df = df.sort_values("step").drop_duplicates(subset=["step"], keep="first")
    else:
        # no agent columns: collapse to one row per step
        df = df.sort_values("step").drop_duplicates(subset=["step"], keep="first")

    # Ensure exactly one row per step now
    df = df.sort_values("step").drop_duplicates(subset=["step"], keep="first")

    out_file = OUT_DIR / f"{ep_id}.jsonl"
    with out_file.open("w") as f:
        for _, row in df.iterrows():
            r = row.to_dict()
            r["episode_id"] = ep_id
            text = build_caption(r)
            step = _safe_int(r.get("step", 0), 0)
            f.write(json.dumps({"episode_id": ep_id, "step": step, "text": text}) + "\n")

    return len(df)

def main():
    count_eps = 0
    count_rows = 0
    for line in MANIFEST.read_text().splitlines():
        if not line.strip(): continue
        item = json.loads(line)
        n = process_episode(item)
        if n:
            count_eps += 1
            count_rows += n
            if count_eps % 50 == 0:
                print(f"[..] {count_eps} episodes, {count_rows} caption rows")
    print(f"[ok] captions -> {OUT_DIR}  ({count_eps} episodes, {count_rows} rows)")

if __name__ == "__main__":
    main()


[ok] captions -> /Users/nginkimlong/Documents/PHD/Exchange Program (SEED)/Ambulance_EGO_4500/ambulance_dataset_fast_150_espisode_cpu_30_senario/captions  (30 episodes, 243508 rows)
